In [ ]:
#!/usr/bin/env python3
"""
Object Recognition Training and Testing Script -- merged version.

Base: the notebook-style CompleteObjectRecognitionSystem script.

Changes applied vs the original notebook script:
  1. detect_objects(): proper IoU-based NMS (torchvision.ops.nms) instead of
     "keep only the largest box". Fixes duplicate/overlapping boxes on the
     same physical object seen in the spark-robot test image, while still
     allowing multiple DISTINCT objects in frame to each get a box.
  2. yolo_conf_threshold raised 0.01 -> 0.25 to stop pulling in noise-level
     detections at the source.
  3. MLflow experiment tracking wired back in (was present in the original
     edge_training.py reference, missing from the notebook script): each
     class-training call and the overall RF training run are logged as
     MLflow runs with params/metrics, and the RF model + feature extractor
     are registered.
  4. Train/validation split added to train_model() -- previously it trained
     and evaluated on the exact same data, so "Training Accuracy" was not a
     real generalization estimate.
  5. evaluate_model() restored -- computes accuracy + unknown-rate against a
     held-out test set with ground-truth parsed from filenames, logged as
     its own MLflow run.
  6. Model directory is now version-aware (auto-increments, e.g. models/v5,
     models/v6, ...) instead of a hardcoded "models/v4" -- mirrors the
     get_latest_model_version() pattern from edge_training.py.

Explicitly NOT changed (per instruction):
  - deploy_to_jetson() is not included. Use your existing auto_deploy.py /
    manual scp flow for that.
  - Minimum object bbox size filter left as-is (w > 30 and h > 30), not
    reverted to the face-recognition script's w > 60 and h > 60.
"""

import cv2
import numpy as np
import time
import os
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.ops import nms
from PIL import Image
import random
import shutil
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.pytorch
from mlflow.tracking import MlflowClient
import json
import pickle
from datetime import datetime
import joblib
import glob
from IPython.display import display, Image as IPImage
import warnings
warnings.filterwarnings('ignore')


class CompleteObjectRecognitionSystem:
    def __init__(self, images_dir="images", models_dir=None,
                 mlflow_uri="sqlite:///mlflow_edge.db",
                 mlflow_experiment="object_recognition_training"):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # --- MLflow setup ---
        mlflow.set_tracking_uri(mlflow_uri)
        mlflow.set_experiment(mlflow_experiment)
        self.mlflow_client = MlflowClient()
        self.registered_rf_name = "object_recognition_rf_model"
        self.registered_fe_name = "object_recognition_feature_extractor"

        # Directories
        self.images_dir = images_dir
        self.few_shot_dir = "few_shot_examples"

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.few_shot_dir, exist_ok=True)

        # --- Version-aware model dir (feature 5) ---
        self.current_model_version = self._get_latest_model_version()
        if models_dir is None:
            models_dir = f"models/v{self.current_model_version}"
        self.models_dir = models_dir
        os.makedirs(self.models_dir, exist_ok=True)

        # Models and data
        self.yolo_model = None
        self.feature_extractor = None
        self.rf_model = None
        self.face_database = {}
        self.face_features = {}
        self.label_encoder = {}
        self.reverse_label_encoder = {}

        # Parameters
        self.confidence_threshold = 0.2
        self.min_examples = 2

        # --- NMS-related params (was yolo_conf_threshold = 0.01) ---
        self.yolo_conf_threshold = 0.25
        self.nms_iou_threshold = 0.45

        # RF parameters
        self.rf_params = {
            'n_estimators': 50,
            'max_depth': 6,
            'min_samples_split': 2,
            'min_samples_leaf': 1,
            'random_state': 42,
            'class_weight': 'balanced'
        }

        # Load models
        self.load_yolo_model()
        self.load_feature_extractor()
        self.load_existing_model()

    # ------------------------------------------------------------------
    # Versioning (feature 5)
    # ------------------------------------------------------------------
    def _get_latest_model_version(self):
        """Get the latest model version from MLflow registry, mirroring
        edge_training.py's get_latest_model_version(). Falls back to
        scanning the local models/ directory if MLflow has no registered
        versions yet (e.g. first run)."""
        try:
            models = self.mlflow_client.search_registered_models(
                filter_string="name='object_recognition_rf_model'"
            )
            if models:
                latest_version = max([
                    int(v.version) for model in models
                    for v in model.latest_versions
                ])
                if latest_version > 0:
                    return latest_version
        except Exception:
            pass

        # Fallback: scan local models/vN directories
        if os.path.isdir("models"):
            existing = [
                d for d in os.listdir("models")
                if d.startswith("v") and d[1:].isdigit()
            ]
            if existing:
                return max(int(d[1:]) for d in existing) + 1
        return 1

    # ------------------------------------------------------------------
    # Model loading
    # ------------------------------------------------------------------
    def load_yolo_model(self):
        print("Loading YOLO model...")
        try:
            from ultralytics import YOLO
            self.yolo_model = YOLO("yolov8n.pt")
            print("YOLO model loaded successfully!")
        except Exception as e:
            print(f"Error loading YOLO: {e}")

    def load_feature_extractor(self):
        print("Loading feature extractor...")
        try:
            model = torchvision.models.mobilenet_v3_small(
                weights=torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
            )
            self.feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
            self.feature_extractor.to(self.device)
            self.feature_extractor.eval()
            print("Feature extractor loaded successfully!")
        except Exception as e:
            print(f"Error loading feature extractor: {e}")

    def load_existing_model(self):
        try:
            db_path = os.path.join(self.models_dir, "face_database.json")
            features_path = os.path.join(self.models_dir, "face_features.pkl")
            rf_path = os.path.join(self.models_dir, "random_forest_model.pkl")
            encoder_path = os.path.join(self.models_dir, "label_encoder.pkl")

            if all(os.path.exists(p) for p in [db_path, features_path, rf_path, encoder_path]):
                with open(db_path, 'r') as f:
                    self.face_database = json.load(f)
                with open(features_path, 'rb') as f:
                    self.face_features = pickle.load(f)
                self.rf_model = joblib.load(rf_path)
                with open(encoder_path, 'rb') as f:
                    encoders = pickle.load(f)
                    self.label_encoder = encoders['label_encoder']
                    self.reverse_label_encoder = encoders['reverse_label_encoder']

                print(f"Loaded existing model with {len(self.face_database)} classes:")
                for class_name in self.label_encoder.keys():
                    print(f"  - {class_name}")
                return True
        except Exception as e:
            print(f"No existing model found or error loading: {e}")
        return False

    # ------------------------------------------------------------------
    # Detection (NMS fix -- changes 1 & 2)
    # ------------------------------------------------------------------
    def detect_objects(self, frame, debug=False):
        """Detect objects using YOLO with proper IoU-based NMS, with
        fallback to a center-region box when nothing survives."""
        objects = []

        if self.yolo_model is None:
            if debug:
                print("YOLO model not available, using fallback")
        else:
            try:
                results = self.yolo_model(
                    frame, verbose=False, conf=self.yolo_conf_threshold
                )

                raw_boxes = []
                raw_scores = []

                for result in results:
                    boxes = result.boxes
                    if boxes is None:
                        continue
                    for box in boxes:
                        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]
                        conf = float(box.conf.cpu().numpy()[0])
                        w, h = x2 - x1, y2 - y1

                        if w > 20 and h > 20:
                            raw_boxes.append([x1, y1, x2, y2])
                            raw_scores.append(conf)

                if debug:
                    print(f"  Pre-NMS: {len(raw_boxes)} candidate box(es)")

                if raw_boxes:
                    boxes_t = torch.tensor(raw_boxes, dtype=torch.float32)
                    scores_t = torch.tensor(raw_scores, dtype=torch.float32)

                    keep_idx = nms(boxes_t, scores_t, self.nms_iou_threshold)

                    if debug:
                        print(f"  Post-NMS: {len(keep_idx)} box(es) kept "
                              f"(iou_threshold={self.nms_iou_threshold})")

                    for i in keep_idx.tolist():
                        x1, y1, x2, y2 = raw_boxes[i]
                        x, y = int(x1), int(y1)
                        w, h = int(x2 - x1), int(y2 - y1)
                        objects.append({
                            "box": (x, y, w, h),
                            "confidence": raw_scores[i],
                        })
                        if debug:
                            print(f"    kept box: {w}x{h} at ({x},{y}) "
                                  f"conf={raw_scores[i]:.3f}")

            except Exception as e:
                if debug:
                    print(f"YOLO detection error: {e}")

        if len(objects) == 0:
            if debug:
                print("Using center region fallback")
            h, w = frame.shape[:2]
            center_x, center_y = w // 2, h // 2
            fallback_w, fallback_h = int(w * 0.8), int(h * 0.8)
            fallback_x = max(0, center_x - fallback_w // 2)
            fallback_y = max(0, center_y - fallback_h // 2)

            objects.append({
                "box": (fallback_x, fallback_y, fallback_w, fallback_h),
                "confidence": 0.5,
            })

        return objects

    # ------------------------------------------------------------------
    # Feature extraction
    # ------------------------------------------------------------------
    def extract_features(self, object_img):
        if self.feature_extractor is None:
            gray = cv2.cvtColor(object_img, cv2.COLOR_BGR2GRAY)
            hist = cv2.calcHist([gray], [0], None, [64], [0, 256])
            hist = cv2.normalize(hist, hist).flatten()
            return hist

        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        try:
            input_tensor = transform(object_img).unsqueeze(0).to(self.device)
            with torch.no_grad():
                features = self.feature_extractor(input_tensor)
                features = features.squeeze().cpu().numpy()
            return features
        except Exception as e:
            print(f"Feature extraction error: {e}")
            gray = cv2.cvtColor(object_img, cv2.COLOR_BGR2GRAY)
            hist = cv2.calcHist([gray], [0], None, [64], [0, 256])
            return cv2.normalize(hist, hist).flatten()

    # ------------------------------------------------------------------
    # Training data prep
    # ------------------------------------------------------------------
    def prepare_training_data(self):
        X = []
        y = []

        unique_classes = list(set(self.face_database.values()))
        unique_classes = [c for c in unique_classes if c != "Unknown"]

        if not unique_classes:
            return np.array([]), np.array([])

        unique_classes.append("Unknown")

        self.label_encoder = {cls: idx for idx, cls in enumerate(unique_classes)}
        self.reverse_label_encoder = {idx: cls for cls, idx in self.label_encoder.items()}

        for obj_id, features_list in self.face_features.items():
            obj_name = self.face_database.get(obj_id, "Unknown")
            if obj_name not in self.label_encoder:
                continue
            obj_label = self.label_encoder[obj_name]
            for features in features_list:
                X.append(features.flatten())
                y.append(obj_label)

        if len(X) > 0:
            X_known = np.array(X)
            unknown_label = self.label_encoder["Unknown"]
            num_unknowns = max(1, len(X) // 6)

            for _ in range(num_unknowns):
                base_idx = np.random.randint(0, len(X_known))
                base_features = X_known[base_idx].copy()
                noise_scale = np.std(base_features) * 0.5
                noise = np.random.normal(0, noise_scale, base_features.shape)
                synthetic_unknown = base_features + noise
                X.append(synthetic_unknown)
                y.append(unknown_label)

        print(f"Training data prepared: {len(X)} samples")
        class_counts = np.bincount(y)
        for i, count in enumerate(class_counts):
            if i in self.reverse_label_encoder:
                print(f"  {self.reverse_label_encoder[i]}: {count} samples")

        return np.array(X), np.array(y)

    # ------------------------------------------------------------------
    # Training (feature 3: MLflow tracking, feature 4: train/val split)
    # ------------------------------------------------------------------
    def train_model(self):
        """Train Random Forest model with a held-out validation split and
        full MLflow tracking (params, metrics, registered model)."""
        print("Training Random Forest model...")

        X, y = self.prepare_training_data()

        if len(X) == 0:
            print("No training data available!")
            return False

        with mlflow.start_run(
            run_name=f"rf_training_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        ):
            mlflow.log_params(self.rf_params)
            mlflow.log_param("num_classes", len(self.label_encoder))
            mlflow.log_param("num_samples", len(X))
            mlflow.log_param("confidence_threshold", self.confidence_threshold)
            mlflow.log_param("yolo_conf_threshold", self.yolo_conf_threshold)
            mlflow.log_param("nms_iou_threshold", self.nms_iou_threshold)
            mlflow.log_param("device", self.device)
            mlflow.log_param("model_version", self.current_model_version)

            try:
                if len(X) > 10:
                    X_train, X_val, y_train, y_val = train_test_split(
                        X, y, test_size=0.2, random_state=42, stratify=y
                    )
                else:
                    print("Dataset too small for a stratified split "
                          f"({len(X)} samples) -- training accuracy will be "
                          "reported on the training set itself, not a true "
                          "generalization estimate.")
                    X_train, X_val, y_train, y_val = X, X, y, y
            except ValueError as e:
                print(f"Stratified split failed ({e}), falling back to train==val")
                X_train, X_val, y_train, y_val = X, X, y, y

            self.rf_model = RandomForestClassifier(**self.rf_params)
            self.rf_model.fit(X_train, y_train)

            train_pred = self.rf_model.predict(X_train)
            train_accuracy = accuracy_score(y_train, train_pred)
            mlflow.log_metric("train_accuracy", train_accuracy)
            print(f"Training Accuracy: {train_accuracy:.3f}")

            val_pred = self.rf_model.predict(X_val)
            val_accuracy = accuracy_score(y_val, val_pred)
            mlflow.log_metric("val_accuracy", val_accuracy)
            print(f"Validation Accuracy: {val_accuracy:.3f}")

            target_names = [self.reverse_label_encoder[i]
                             for i in sorted(self.reverse_label_encoder.keys())]
            print("\nValidation Classification Report:")
            print(classification_report(y_val, val_pred, target_names=target_names,
                                         zero_division=0))

            self.save_model()

            mlflow.log_artifact(os.path.join(self.models_dir, "face_database.json"))
            mlflow.log_artifact(os.path.join(self.models_dir, "random_forest_model.pkl"))
            mlflow.log_artifact(os.path.join(self.models_dir, "label_encoder.pkl"))

            mlflow.sklearn.log_model(
                sk_model=self.rf_model,
                name="random_forest_classifier",
                registered_model_name=self.registered_rf_name
            )

            if self.feature_extractor:
                mlflow.pytorch.log_model(
                    pytorch_model=self.feature_extractor,
                    name="feature_extractor",
                    registered_model_name=self.registered_fe_name
                )

        print("Model trained, evaluated, and saved successfully!")
        return True

    def save_model(self):
        db_path = os.path.join(self.models_dir, "face_database.json")
        with open(db_path, 'w') as f:
            json.dump(self.face_database, f)

        features_path = os.path.join(self.models_dir, "face_features.pkl")
        with open(features_path, 'wb') as f:
            pickle.dump(self.face_features, f)

        if self.rf_model:
            rf_path = os.path.join(self.models_dir, "random_forest_model.pkl")
            joblib.dump(self.rf_model, rf_path)

            encoder_path = os.path.join(self.models_dir, "label_encoder.pkl")
            encoders = {
                'label_encoder': self.label_encoder,
                'reverse_label_encoder': self.reverse_label_encoder
            }
            with open(encoder_path, 'wb') as f:
                pickle.dump(encoders, f)

    def train_on_class(self, class_name, debug=True):
        """Train model on a specific class. Logs a per-class MLflow run
        (params + extraction success rate), mirroring the per-person
        tracking in edge_training.py's train_few_shot_model()."""
        class_dir = os.path.join(self.images_dir, class_name)

        if not os.path.exists(class_dir):
            print(f"Error: Directory {class_dir} not found!")
            return False

        image_files = []
        for ext in ['*.jpg', '*.jpeg', '*.png']:
            image_files.extend(glob.glob(os.path.join(class_dir, ext)))

        if not image_files:
            print(f"No images found in {class_dir}")
            return False

        print(f"Training on {class_name} with {len(image_files)} images...")

        if self.face_database:
            new_id = str(max(int(k) for k in self.face_database.keys()) + 1)
        else:
            new_id = "0"

        with mlflow.start_run(
            run_name=f"class_training_{class_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        ):
            mlflow.log_param("class_name", class_name)
            mlflow.log_param("num_images", len(image_files))
            mlflow.log_param("min_examples", self.min_examples)

            successful_extractions = 0
            feature_list = []

            for img_path in image_files:
                if debug:
                    print(f"Processing: {os.path.basename(img_path)}")

                frame = cv2.imread(img_path)
                if frame is None:
                    continue

                objects = self.detect_objects(frame, debug=debug)

                if objects:
                    largest_obj = max(objects, key=lambda x: x["box"][2] * x["box"][3])
                    x, y, w, h = largest_obj["box"]

                    if w > 30 and h > 30:
                        object_roi = frame[y:y+h, x:x+w]
                        features = self.extract_features(object_roi)
                        feature_list.append(features)
                        successful_extractions += 1

                        if debug:
                            print(f"  Extracted features: {features.shape}")

            extraction_rate = successful_extractions / len(image_files) if image_files else 0
            mlflow.log_metric("successful_extractions", successful_extractions)
            mlflow.log_metric("extraction_success_rate", extraction_rate)

            if successful_extractions >= self.min_examples:
                self.face_database[new_id] = class_name
                self.face_features[new_id] = feature_list
                print(f"Successfully processed {class_name}: {successful_extractions} examples")
                return True
            else:
                print(f"Insufficient examples for {class_name}: "
                      f"got {successful_extractions}, need {self.min_examples}")
                return False

    # ------------------------------------------------------------------
    # Recognition / inference
    # ------------------------------------------------------------------
    def recognize_object(self, object_roi, debug=False):
        if self.rf_model is None:
            return {"name": "Unknown", "confidence": 0, "object_id": None}

        query_features = self.extract_features(object_roi).flatten().reshape(1, -1)

        try:
            probabilities = self.rf_model.predict_proba(query_features)[0]
            predicted_class = np.argmax(probabilities)
            confidence = probabilities[predicted_class]

            if debug:
                print(f"Prediction probabilities: "
                      f"{dict(zip(self.reverse_label_encoder.values(), probabilities))}")
                print(f"Predicted class: {predicted_class}, confidence: {confidence:.3f}")

            object_name = self.reverse_label_encoder.get(predicted_class, "Unknown")

            if object_name != "Unknown" and confidence < self.confidence_threshold:
                if debug:
                    print(f"Confidence {confidence:.3f} below threshold "
                          f"{self.confidence_threshold}, returning Unknown")
                return {"name": "Unknown", "confidence": confidence * 100, "object_id": None}

            object_id = None
            for oid, name in self.face_database.items():
                if name == object_name:
                    object_id = oid
                    break

            return {
                "name": object_name,
                "confidence": confidence * 100,
                "object_id": object_id
            }

        except Exception as e:
            print(f"Recognition error: {e}")
            return {"name": "Error", "confidence": 0, "object_id": None}

    def test_image(self, image_path, debug=True, show_image=True):
        print(f"\n--- Testing: {os.path.basename(image_path)} ---")

        frame = cv2.imread(image_path)
        if frame is None:
            print("Error loading image!")
            return None

        objects = self.detect_objects(frame, debug=debug)
        print(f"Detected {len(objects)} object(s)")

        results = []
        result_frame = frame.copy()

        for i, obj in enumerate(objects):
            x, y, w, h = obj["box"]

            if w > 30 and h > 30:
                object_roi = frame[y:y+h, x:x+w]
                recognition = self.recognize_object(object_roi, debug=debug)

                result = {"bbox": (x, y, w, h), "recognition": recognition}
                results.append(result)

                print(f"Object {i}: {recognition['name']} ({recognition['confidence']:.1f}%)")

                color = (0, 255, 0) if recognition['name'] != 'Unknown' else (0, 0, 255)
                cv2.rectangle(result_frame, (x, y), (x+w, y+h), color, 2)
                label = f"{recognition['name']}: {recognition['confidence']:.1f}%"
                cv2.putText(result_frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, color, 2)

        if show_image:
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(result_frame, cv2.COLOR_BGR2RGB))
            plt.title(f"Recognition Result: {os.path.basename(image_path)}")
            plt.axis('off')
            plt.show()

        return results

    # ------------------------------------------------------------------
    # Evaluation (feature 4: restored from edge_training.py)
    # ------------------------------------------------------------------
    def evaluate_model(self, test_image_paths):
        """Evaluate model performance against a held-out test set.
        Ground truth is parsed from the filename (format: classname_*.jpg,
        matching the convention already used in test_images_path below).
        Logs accuracy and unknown-rate as an MLflow run."""
        with mlflow.start_run(
            run_name=f"model_evaluation_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        ):
            mlflow.log_param("num_test_images", len(test_image_paths))
            mlflow.log_param("model_version", self.current_model_version)

            correct_predictions = 0
            total_predictions = 0
            unknown_count = 0

            for img_path in test_image_paths:
                filename = os.path.basename(img_path)
                ground_truth = filename.rsplit('_', 1)[0] if '_' in filename else filename.split('.')[0]

                frame = cv2.imread(img_path)
                if frame is None:
                    continue

                objects = self.detect_objects(frame)
                if objects:
                    largest_obj = max(objects, key=lambda o: o["box"][2] * o["box"][3])
                    x, y, w, h = largest_obj["box"]
                    object_roi = frame[y:y+h, x:x+w]

                    prediction = self.recognize_object(object_roi)

                    total_predictions += 1
                    if prediction["name"] == ground_truth:
                        correct_predictions += 1
                    if prediction["name"] == "Unknown":
                        unknown_count += 1

            accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
            unknown_rate = unknown_count / total_predictions if total_predictions > 0 else 0

            mlflow.log_metric("eval_accuracy", accuracy)
            mlflow.log_metric("eval_unknown_rate", unknown_rate)
            mlflow.log_metric("eval_total_predictions", total_predictions)

            print(f"Model Evaluation: Accuracy={accuracy:.2f}, Unknown Rate={unknown_rate:.2f}, "
                  f"N={total_predictions}")
            return accuracy


# ==================== Module-level helpers ====================

print("Initializing Complete Object Recognition System...")
system = CompleteObjectRecognitionSystem()


def train_all_classes(class_names):
    """Train on multiple classes, then fit + evaluate the RF model."""
    print("=== TRAINING PHASE ===")

    success_count = 0
    for class_name in class_names:
        print(f"\n--- Training on {class_name} ---")
        if system.train_on_class(class_name, debug=True):
            success_count += 1

    if success_count > 0:
        print(f"\n--- Training Random Forest on {success_count} classes ---")
        system.train_model()
        return True
    else:
        print("No classes successfully processed!")
        return False


def test_images(test_image_paths, debug=False):
    """Test on multiple images (qualitative -- prints predictions, no
    ground-truth accuracy). Use evaluate_model() for a real accuracy number."""
    print("\n=== TESTING PHASE ===")

    all_results = {}
    prediction_counts = {}

    for image_path in test_image_paths:
        results = system.test_image(image_path, debug=debug, show_image=True)
        all_results[image_path] = results

        if results:
            for result in results:
                name = result['recognition']['name']
                prediction_counts[name] = prediction_counts.get(name, 0) + 1

    print("\n=== PREDICTION SUMMARY ===")
    for name, count in sorted(prediction_counts.items()):
        print(f"{name}: {count} predictions")

    return all_results


# ==================== USAGE EXAMPLES ====================

class_names = ["spark-robot", "Leave-Warning-Sign", "Fire-extinguisher",
               "High_Voltage_Sign", "amrit", "tomasz", "dove-cream", "oranges"]

# Uncomment to train:
train_success = train_all_classes(class_names)

test_images_path = [
    "images/elon/elon_3.jpg",
    "images/spark-robot/spark-robot_2.jpg",
    "images/tomasz/tomasz_3.jpg",
    "images/amrit/amrit_2.jpg",
    "images/dove-cream/dove-cream_2.jpg",
]

#Uncomment to test qualitatively:
test_results = test_images(test_images_path, debug=True)

#Uncomment to get a real accuracy number against ground truth in filenames:
accuracy = system.evaluate_model(test_images_path)

print("\n" + "="*50)
print("SYSTEM READY!")
print("="*50)
print("To use:")
print("1. Place training images in folders: images/spark-robot/, images/Leave-Warning-Sign/, etc.")
print("2. Run: train_success = train_all_classes(class_names)")
print("3. Add test image paths to test_images_path list (format: classname_N.jpg)")
print("4. Run: test_results = test_images(test_images_path, debug=True)  # qualitative")
print("5. Run: accuracy = system.evaluate_model(test_images_path)        # quantitative")
print("="*50)